# Calculate Regrowth Fractions - ΔpqsL Mutant

This notebook calculates regrowth fractions for ΔpqsL mutant across all replicates and generates the intermediate CSV files needed for plotting.

**Analysis includes:**
- Calculating mean cells per bin from wild-type fluorescence data (reference)
- Creating spatial bins along X and Y axes
- Computing regrowth fractions per chamber, per replicate, and averaged across replicates
- Generating visualization of spatial regrowth patterns

**Outputs:**
- `1_first_colonies_pqsL_merged.csv`: Merged colony data from all replicates
- `1_regrowth_fractions_pqsL_merged.csv`: Mean regrowth fractions across replicates
- `1_regrowth_fractions_pqsL_per_replicate.csv`: Regrowth fractions per replicate
- `1_regrowth_fractions_pqsL_per_chamber.csv`: Regrowth fractions per chamber

## Import Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.gridspec as gridspec
import os

## Analyze Mean Cells Per Bin

Calculate the mean number of cells per bin from wild-type fluorescence data to use as normalization reference (b_mean)

In [ ]:
# Parameters
pixel_to_um = 0.065
frame_number = 0
num_bins = 10
max_distance = 50  # µm

# Bin edges in µm
y_bins = np.linspace(0, max_distance, num_bins + 1)
bin_width = (y_bins[-1] - y_bins[0]) / num_bins

# Read fluorescence data using relative paths
replicate_sources = [
    ('rep1', '../../../../Figure1/1E_wt/analysis_code/1_fluo_binary_wt_rep1.csv'),
    ('rep2', '../../../../Figure1/1E_wt/analysis_code/1_fluo_binary_wt_rep2.csv'),
    ('rep3', '../../../../Figure1/1E_wt/analysis_code/1_fluo_binary_wt_rep3.csv'),
]

# Build cell dataframe
cell_frames = []
for rep_label, csv_path in replicate_sources:
    df = pd.read_csv(csv_path)
    df = df[df['frame_number'] == frame_number].copy()
    if df.empty:
        continue

    # Convert pixels to µm
    df['x_um'] = df['x'] * pixel_to_um
    df['y_um'] = df['y'] * pixel_to_um

    # Get cell centers (one row per cell)
    cell_df = (df
               .groupby(['pos', 'label'], as_index=False)
               .agg(x_um=('x_um', 'mean'),
                    y_um=('y_um', 'mean')))
    
    cell_df['replicate'] = rep_label
    cell_frames.append(cell_df)

cells = pd.concat(cell_frames, ignore_index=True)

# Filter to 0-40 µm range
cells = cells[(cells['x_um'] >= 0) & (cells['x_um'] <= max_distance) & 
              (cells['y_um'] >= 0) & (cells['y_um'] <= max_distance)].copy()

print(f"Total cells analyzed: {len(cells)}")

# Assign y bins
cells['y_bin'] = pd.cut(cells['y_um'], bins=y_bins, include_lowest=True)
cells = cells.dropna(subset=['y_bin'])
cells['bin_left'] = cells['y_bin'].apply(lambda iv: float(iv.left))

# Count cells per chamber per bin
chamber_bin_counts = (cells
                      .groupby(['replicate', 'pos', 'bin_left'], as_index=False)
                      .size()
                      .rename(columns={'size': 'cell_count'}))

# Step 1: Mean across chambers within each replicate
replicate_bin_stats = (chamber_bin_counts
                       .groupby(['replicate', 'bin_left'], as_index=False)
                       .agg(mean_cells_per_bin=('cell_count', 'mean')))

# Step 2: Mean across replicates
bin_stats = (replicate_bin_stats
             .groupby('bin_left', as_index=False)
             .agg(mean_cells_per_bin=('mean_cells_per_bin', 'mean')))

bin_stats['bin_left'] = bin_stats['bin_left'].astype(float)

# Calculate overall mean (mean of bin means)
b_mean = bin_stats['mean_cells_per_bin'].mean()
b_std = bin_stats['mean_cells_per_bin'].std()

print(f"\nMean of mean cells per bin (across {num_bins} bins): {b_mean:.2f} ± {b_std:.2f}")
print(f"\nThis b_mean value will be used for regrowth fraction calculation.")

## Create Bins Along X and Y

Define spatial bins for analyzing colony distribution

In [ ]:
# Define bin edges (10 bins from 0 to 50 µm)
x_bins = np.linspace(0, max_distance, num_bins + 1)

bin_width = (x_bins[1] - x_bins[0])

print(f"Bin configuration for colony analysis:")
print(f"  Number of bins: {num_bins}")
print(f"  Range: 0 to {max_distance} µm")
print(f"  Bin width: {bin_width:.2f} µm")
print(f"\nx_bins: {x_bins}")
print(f"y_bins: {y_bins}")

Bin configuration for colony analysis:
  Number of bins: 10
  Range: 0 to 50 µm
  Bin width: 5.00 µm

x_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]
y_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]


## Calculate Regrowth Fractions

Process colony data from all replicates and compute regrowth fractions per chamber, per replicate, and averaged

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Assumes x_bins, y_bins, bin_width, pixel_to_um, and b_mean are already defined


# Use relative paths for colony data
replicates = [
    {
        'colonies_path': '0_first_colonies_pqsL_rep1.csv',
        'replicate': 'rep1'
    },
    {
        'colonies_path': '0_first_colonies_pqsL_rep2.csv',
        'replicate': 'rep2'
    },
    {
        'colonies_path': '0_first_colonies_pqsL_rep3.csv',
        'replicate': 'rep3'
    }
]

# Save outputs with relative paths
OUTPUT_COLONIES = '1_first_colonies_pqsL_merged.csv'
OUTPUT_REGROWTH_FINAL = '1_regrowth_fractions_pqsL_merged.csv'
OUTPUT_PER_REP_COMBINED = '1_regrowth_fractions_pqsL_per_replicate.csv'
OUTPUT_PER_CHAMBER = '1_regrowth_fractions_pqsL_per_chamber.csv'

print(f"Using b_mean = {b_mean:.2f} for normalization\n")

# =============================================================================
# STEP 1 — Compute per-chamber regrowth fractions
# =============================================================================
all_colonies = []
regrowth_per_chamber = []
total_chambers_per_replicate = {}

for rep in replicates:
    path = rep["colonies_path"]
    rep_name = rep["replicate"]

    if not os.path.exists(path):
        print(f"Skipping {rep_name} (file not found)")
        continue

    df = pd.read_csv(path)
    total_chambers = df["position"].nunique()   # total chambers regardless of regrowth
    total_chambers_per_replicate[rep_name] = total_chambers

    print(f"{rep_name}: {total_chambers} chambers total")
    positions = sorted(df["position"].dropna().unique())
    print("Chambers:", ", ".join(positions))
    print("-" * 80)

    # Keep only regrowing colonies
    colonies = df[df["colony_id"].notna()].copy()
    if colonies.empty:
        print(f"{rep_name}: no regrowth detected\n")
        continue

    # Convert frames to hours and filter by time
    colonies["hours"] = colonies["frame"] * (5 / 60)
    colonies = colonies[(colonies["hours"] <= 20) & (colonies["colony_id"] < 10000)].copy()
    if colonies.empty:
        print(f"{rep_name}: no colonies after filtering\n")
        continue

    colonies["replicate"] = rep_name
    colonies["x_um"] = colonies["x"] * pixel_to_um
    colonies["y_um"] = colonies["y"] * pixel_to_um
    colonies["x_bin"] = pd.cut(colonies["x_um"], bins=x_bins, include_lowest=True)
    colonies["y_bin"] = pd.cut(colonies["y_um"], bins=y_bins, include_lowest=True)

    # --- Count colonies per chamber per bin (Y-axis) ---
    chamber_bin_counts_y = (
        colonies.groupby(["position", "y_bin"])
        .size()
        .reset_index(name="colony_count")
        .assign(
            replicate=rep_name,
            axis="y",
            bin_left=lambda df: df["y_bin"].apply(lambda x: float(x.left) if pd.notna(x) else None)
        )
        .dropna(subset=["bin_left"])
    )
    # Clip negative bin edges to 0
    chamber_bin_counts_y["bin_left"] = chamber_bin_counts_y["bin_left"].astype(float).clip(lower=0)
    chamber_bin_counts_y["regrowth_fraction"] = chamber_bin_counts_y["colony_count"] / b_mean

    # --- Count colonies per chamber per bin (X-axis) ---
    chamber_bin_counts_x = (
        colonies.groupby(["position", "x_bin"])
        .size()
        .reset_index(name="colony_count")
        .assign(
            replicate=rep_name,
            axis="x",
            bin_left=lambda df: df["x_bin"].apply(lambda x: float(x.left) if pd.notna(x) else None)
        )
        .dropna(subset=["bin_left"])
    )
    # Clip negative bin edges to 0
    chamber_bin_counts_x["bin_left"] = chamber_bin_counts_x["bin_left"].astype(float).clip(lower=0)
    chamber_bin_counts_x["regrowth_fraction"] = chamber_bin_counts_x["colony_count"] / b_mean

    # Store per-chamber data
    regrowth_per_chamber.append(
        chamber_bin_counts_y[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )
    regrowth_per_chamber.append(
        chamber_bin_counts_x[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )

    all_colonies.append(colonies)

# Combine all replicate data
if regrowth_per_chamber:
    regrowth_per_chamber_df = pd.concat(regrowth_per_chamber, ignore_index=True)
else:
    regrowth_per_chamber_df = pd.DataFrame(columns=["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"])

# =============================================================================
# CRITICAL FIX: Add all chambers with zero regrowth
# =============================================================================
# Create complete grid of all replicates × all positions × all bins × both axes
bin_width = x_bins[1] - x_bins[0]
all_bin_lefts = np.arange(x_bins[0], x_bins[-1], bin_width)

complete_chamber_grid = []
for rep in replicates:
    path = rep["colonies_path"]
    rep_name = rep["replicate"]
    
    if not os.path.exists(path):
        continue
    
    # Get ALL positions from the original file (including those without regrowth)
    df = pd.read_csv(path)
    all_positions = df["position"].dropna().unique()
    
    # Create grid for this replicate
    for position in all_positions:
        for axis in ['x', 'y']:
            for bin_left in all_bin_lefts:
                complete_chamber_grid.append({
                    'replicate': rep_name,
                    'position': position,
                    'axis': axis,
                    'bin_left': bin_left
                })

complete_chamber_grid_df = pd.DataFrame(complete_chamber_grid)

# Merge with actual data, filling missing values with 0
regrowth_per_chamber_df = complete_chamber_grid_df.merge(
    regrowth_per_chamber_df[['replicate', 'position', 'axis', 'bin_left', 'colony_count', 'regrowth_fraction']],
    on=['replicate', 'position', 'axis', 'bin_left'],
    how='left'
)
regrowth_per_chamber_df['colony_count'] = regrowth_per_chamber_df['colony_count'].fillna(0)
regrowth_per_chamber_df['regrowth_fraction'] = regrowth_per_chamber_df['regrowth_fraction'].fillna(0)

print(f"\n✅ Complete chamber grid created:")
print(f"   Total chamber-bin combinations: {len(regrowth_per_chamber_df)}")
print(f"   Chambers with regrowth: {(regrowth_per_chamber_df['regrowth_fraction'] > 0).sum()}")
print(f"   Chambers without regrowth: {(regrowth_per_chamber_df['regrowth_fraction'] == 0).sum()}")

# Save merged colonies data
merged_colonies_df = pd.concat(all_colonies, ignore_index=True) if all_colonies else pd.DataFrame()
merged_colonies_df.to_csv(OUTPUT_COLONIES, index=False)
print(f"\n✅ Saved merged colonies to:\n{OUTPUT_COLONIES}")

# Save per-chamber data
regrowth_per_chamber_df.to_csv(OUTPUT_PER_CHAMBER, index=False)
print(f"✅ Saved per-chamber bin table to:\n{OUTPUT_PER_CHAMBER}")

# =============================================================================
# STEP 2 — Normalize per replicate by total chambers (including zeros)
# =============================================================================
replicate_bin_stats = (
    regrowth_per_chamber_df
    .groupby(["replicate", "axis", "bin_left"], as_index=False)
    .agg(total_fraction_in_bin=("regrowth_fraction", "sum"))
)
replicate_bin_stats["total_chambers"] = replicate_bin_stats["replicate"].map(total_chambers_per_replicate)
replicate_bin_stats["regrowth_fraction"] = (
    replicate_bin_stats["total_fraction_in_bin"] / replicate_bin_stats["total_chambers"]
)

# =============================================================================
# STEP 2.5 — Add missing replicates with zeros
# =============================================================================
# Create complete grid of all replicates × all bins × both axes
bin_width = x_bins[1] - x_bins[0]
all_bin_lefts = np.arange(x_bins[0], x_bins[-1], bin_width)
all_replicate_names = [rep['replicate'] for rep in replicates]

complete_grid = []
for rep_name in all_replicate_names:
    for axis in ['x', 'y']:
        for bin_left in all_bin_lefts:
            complete_grid.append({
                'replicate': rep_name,
                'axis': axis,
                'bin_left': bin_left
            })

complete_grid_df = pd.DataFrame(complete_grid)

# Merge with actual data, filling missing values with 0
replicate_bin_stats_complete = complete_grid_df.merge(
    replicate_bin_stats[['replicate', 'axis', 'bin_left', 'regrowth_fraction']],
    on=['replicate', 'axis', 'bin_left'],
    how='left'
)
replicate_bin_stats_complete['regrowth_fraction'] = replicate_bin_stats_complete['regrowth_fraction'].fillna(0)

# Save combined per-replicate table (only selected columns)
replicate_bin_stats_complete.to_csv(OUTPUT_PER_REP_COMBINED, index=False)
print(f"✅ Saved per-replicate bin table to:\n{OUTPUT_PER_REP_COMBINED}")

# =============================================================================
# STEP 3 — Mean across replicates
# =============================================================================
n_replicates = len(replicates)
bin_stats = (
    replicate_bin_stats_complete
    .groupby(["axis", "bin_left"], as_index=False)
    .agg(
        mean_regrowth_fraction=("regrowth_fraction", "mean"),
        std_regrowth_fraction=("regrowth_fraction", "std"),
        n_replicates=("replicate", "nunique")
    )
)
bin_stats["std_regrowth_fraction"] = bin_stats["std_regrowth_fraction"].fillna(0)

# Save final results
bin_stats.to_csv(OUTPUT_REGROWTH_FINAL, index=False)
print(f"✅ Saved regrowth fractions to:\n{OUTPUT_REGROWTH_FINAL}")

# =============================================================================
# PRINT SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("REGROWTH FRACTION PER BIN (mean ± std across replicates)")
print("=" * 80)

for axis in ["x", "y"]:
    print(f"\n{axis.upper()}-AXIS:")
    print("-" * 80)
    axis_stats = bin_stats[bin_stats["axis"] == axis].sort_values("bin_left")

    for _, row in axis_stats.iterrows():
        b0, b1 = row["bin_left"], row["bin_left"] + bin_width
        print(f"  Bin {b0:5.1f}-{b1:5.1f} µm: "
              f"{row['mean_regrowth_fraction']:.6f} ± {row['std_regrowth_fraction']:.6f} "
              f"(n={row['n_replicates']})")

    overall_mean = axis_stats["mean_regrowth_fraction"].mean()
    overall_std = axis_stats["mean_regrowth_fraction"].std()
    print(f"\n  Overall {axis}-axis: {overall_mean:.6f} ± {overall_std:.6f}")

print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(f"Total colonies: {len(merged_colonies_df)}")
print(f"Total chamber-bin combinations: {len(regrowth_per_chamber_df)}")
print(f"Mean regrowth fraction (across all bins): {bin_stats['mean_regrowth_fraction'].mean():.6f} ± {bin_stats['std_regrowth_fraction'].std():.6f}")

Using b_mean = 159.66 for normalization

rep1: 9 chambers total
Chambers: pos15, pos16, pos17, pos18, pos19, pos20, pos21, pos22, pos23
--------------------------------------------------------------------------------
rep1: no regrowth detected

rep2: 9 chambers total
Chambers: pos0, pos1, pos2, pos3, pos5, pos6, pos7, pos8, pos9
--------------------------------------------------------------------------------
rep3: 10 chambers total
Chambers: pos0, pos1, pos2, pos3, pos4, pos5, pos6, pos7, pos8, pos9
--------------------------------------------------------------------------------
rep3: no regrowth detected


✅ Complete chamber grid created:
   Total chamber-bin combinations: 560
   Chambers with regrowth: 7
   Chambers without regrowth: 553

✅ Saved merged colonies to:
1_first_colonies_pqsL_merged.csv
✅ Saved per-chamber bin table to:
1_regrowth_fractions_pqsL_per_chamber.csv
✅ Saved per-replicate bin table to:
1_regrowth_fractions_pqsL_per_replicate.csv
✅ Saved regrowth fractions to:
1